## Dataset provenance & safety notes

- Data comes from `md-emg-python/data/healthy/S0/emg_logs/session_XX.npy` with each file containing stacked acquisition buffers converted to 32-channel (0–31) EMG traces plus a timestamp column (removed here).
- Sessions flagged as "bad" or "disconnected" in `data/notes.txt` are excluded.
- Active detection uses an adaptive RMS threshold; tweak `threshold_sigma`, `min_active_sec`, or `min_rest_sec` if the envelope mask misclassifies events.
- Overshoot: no explicit clipping yet—large spikes will inflate RMS metrics. Consider adding stationarity checks if needed.
- Sampling rate assumed 1 kHz; adjust `FS_HZ` if acquisition configuration changes.


# Comparative EMG Session Analysis

This notebook compares EMG activity for Subject S0 across multiple session types:

- **Session 00** – MVC reference
- **Sessions 05 & 06** – Passive glove trials
- **Sessions 08, 09 & 10** – Active glove trials
- **Sessions 11, 12 & 16** – No glove trials

We'll build the analysis in five stages:

1. **Configuration & Metadata** – imports, data paths, session labels, helper configs (channel indices 0–31, 1 kHz sampling).
2. **Utilities** – loaders, cleaning helpers (convert concatenated buffers to a single array, drop the timestamp column, detect overshoot).
3. **Signal Summaries** – RMS envelopes, active segment extraction via adaptive thresholds, basic stats per channel/condition.
4. **Comparative Visuals** – stacked time plots, condition-averaged RMS heatmaps (4×8 grid + circular arm layout), and condition-level box/violin plots.
5. **Dimensionality Views** – PCA/UMAP on active windows and optional clustering for glove vs no-glove discrimination.

Throughout, we'll keep channel numbering at 0–31 and ignore the timestamp column. Notes from `data/notes.txt` guide which sessions to skip (e.g., bad/overshoot runs).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple
import math
import warnings

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
FS_HZ = 1000  # Sampling rate (Hz) from config/emg_signal_processing.yaml
CHANNEL_IDS = list(range(32))
CHANNEL_GRID = np.array([
    [0, 1, 2, 3, 4, 5, 6, 7],
    [8, 9, 10, 11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20, 21, 22, 23],
    [24, 25, 26, 27, 28, 29, 30, 31],
])
CONDITION_ORDER = ["MVC", "Passive glove", "Active glove", "No glove"]

REPO_ROOT = Path.cwd().resolve()
DATA_ROOT = (REPO_ROOT / "data" / "healthy" / "S0" / "emg_logs").resolve()
assert DATA_ROOT.exists(), f"Data folder not found: {DATA_ROOT}"

SESSION_METADATA = pd.DataFrame([
    {"session": 0, "condition": "MVC", "label": "Max voluntary contraction"},
    {"session": 5, "condition": "Passive glove", "label": "Passive glove (trial A)"},
    {"session": 6, "condition": "Passive glove", "label": "Passive glove (trial B)"},
    {"session": 15, "condition": "Passive glove", "label": "Passive glove (trial C)"},
    {"session": 8, "condition": "Active glove", "label": "Active glove (trial A)"},
    {"session": 9, "condition": "Active glove", "label": "Active glove (trial B)"},
    {"session": 10, "condition": "Active glove", "label": "Active glove (trial C)"},
    {"session": 11, "condition": "No glove", "label": "No glove (trial A)"},
    {"session": 12, "condition": "No glove", "label": "No glove (trial B)"},
    {"session": 16, "condition": "No glove", "label": "No glove (trial C)"},
]).set_index("session")
SESSION_METADATA = SESSION_METADATA.loc[:, ["condition", "label"]]
SESSION_METADATA

,condition,label
session,,
0,MVC,Max voluntary contraction
5,Passive glove,Passive glove (trial A)
6,Passive glove,Passive glove (trial B)
15,Passive glove,Passive glove (trial C)
8,Active glove,Active glove (trial A)
9,Active glove,Active glove (trial B)
10,Active glove,Active glove (trial C)
11,No glove,No glove (trial A)
12,No glove,No glove (trial B)


In [3]:
def load_session(session_id: int, drop_timestamp: bool = True) -> np.ndarray:
    """Load a session file and return a (samples × channels) array."""
    session_path = DATA_ROOT / f"session_{session_id:02d}.npy"
    if not session_path.exists():
        raise FileNotFoundError(f"Missing session file: {session_path}")
    arrays: List[np.ndarray] = []
    with open(session_path, "rb") as handle:
        while True:
            try:
                arrays.append(np.load(handle, allow_pickle=False))
            except (ValueError, EOFError):
                break
    if not arrays:
        raise ValueError(f"No arrays found inside {session_path}")
    data = np.concatenate(arrays, axis=0)
    if data.ndim == 1:
        data = data[:, None]
    if drop_timestamp and data.shape[1] > len(CHANNEL_IDS):
        data = data[:, : len(CHANNEL_IDS)]
    return data.astype(np.float32, copy=False)


def keep_long_runs(mask: np.ndarray, min_samples: int) -> np.ndarray:
    """Remove active runs shorter than `min_samples`."""
    mask = np.asarray(mask, dtype=bool).copy()
    if mask.sum() == 0:
        return mask
    start_idx = None
    for idx, flag in enumerate(mask):
        if flag and start_idx is None:
            start_idx = idx
        elif not flag and start_idx is not None:
            if idx - start_idx < min_samples:
                mask[start_idx:idx] = False
            start_idx = None
    if start_idx is not None and len(mask) - start_idx < min_samples:
        mask[start_idx:] = False
    return mask


def detect_active_regions(
    emg: np.ndarray,
    fs_hz: int = FS_HZ,
    window_sec: float = 0.200,
    threshold_sigma: float = 3.0,
    min_active_sec: float = 0.250,
    min_rest_sec: float = 0.300,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """
    Estimate active vs. rest samples using a rolling RMS envelope.

    Returns
    -------
    active_mask : bool array
        Samples considered active.
    rest_mask : bool array
        Samples considered rest/baseline.
    envelope : float array
        Smoothed global RMS envelope.
    threshold : float
        Activity threshold applied to the envelope.
    """
    emg = np.asarray(emg, dtype=np.float32)
    window = max(int(window_sec * fs_hz), 1)
    min_active = max(int(min_active_sec * fs_hz), 1)
    min_rest = max(int(min_rest_sec * fs_hz), 1)

    composite = np.sqrt(np.mean(np.square(emg), axis=1))
    envelope = (
        pd.Series(composite)
        .rolling(window, min_periods=max(1, window // 2))
        .mean()
        .bfill()
        .ffill()
    )
    baseline = envelope.quantile(0.25)
    noise_level = (envelope - baseline).mad()
    if not np.isfinite(noise_level) or noise_level == 0:
        noise_level = envelope.mad()
    if not np.isfinite(noise_level) or noise_level == 0:
        noise_level = envelope.std()
    if not np.isfinite(noise_level) or noise_level == 0:
        noise_level = max(1e-3, float(np.median(envelope)))
    threshold = float(baseline + threshold_sigma * noise_level)

    active_mask = (envelope > threshold).to_numpy()
    active_mask = keep_long_runs(active_mask, min_active)
    rest_mask = np.logical_not(active_mask)
    secondary_rest = (envelope <= baseline + noise_level).to_numpy()
    if secondary_rest.sum() > rest_mask.sum():
        rest_mask = secondary_rest
    rest_mask = keep_long_runs(rest_mask, min_rest)
    if rest_mask.sum() == 0:
        rest_mask[:min_rest] = True

    return active_mask, rest_mask, envelope.to_numpy(), threshold


@dataclass
class SessionSummary:
    session: int
    condition: str
    label: str
    emg: np.ndarray
    time: np.ndarray
    active_mask: np.ndarray
    rest_mask: np.ndarray
    envelope: np.ndarray
    threshold: float

    @property
    def active_duration(self) -> float:
        return float(self.active_mask.sum() / FS_HZ)

    @property
    def rest_duration(self) -> float:
        return float(self.rest_mask.sum() / FS_HZ)

    def channel_metrics(self) -> pd.DataFrame:
        emg_active = self.emg[self.active_mask] if self.active_mask.any() else np.empty((0, self.emg.shape[1]))
        emg_rest = self.emg[self.rest_mask] if self.rest_mask.any() else np.empty((0, self.emg.shape[1]))

        def safe_rms(block: np.ndarray) -> np.ndarray:
            if block.size == 0:
                return np.full(self.emg.shape[1], np.nan, dtype=np.float32)
            return np.sqrt(np.mean(np.square(block), axis=0))

        def safe_peak(block: np.ndarray) -> np.ndarray:
            if block.size == 0:
                return np.full(self.emg.shape[1], np.nan, dtype=np.float32)
            return np.max(np.abs(block), axis=0)

        rms_active = safe_rms(emg_active)
        rms_rest = safe_rms(emg_rest)
        peak_active = safe_peak(emg_active)
        peak_rest = safe_peak(emg_rest)
        activation_ratio = rms_active / np.clip(rms_rest, 1e-6, None)

        return pd.DataFrame({
            "session": self.session,
            "condition": self.condition,
            "label": self.label,
            "channel": CHANNEL_IDS,
            "rms_active": rms_active,
            "rms_rest": rms_rest,
            "activation_ratio": activation_ratio,
            "peak_active": peak_active,
            "peak_rest": peak_rest,
            "active_duration": self.active_duration,
            "rest_duration": self.rest_duration,
        })


def build_session_summary(session_id: int) -> SessionSummary:
    meta = SESSION_METADATA.loc[session_id]
    emg = load_session(session_id)
    time = np.arange(emg.shape[0], dtype=np.float32) / FS_HZ
    active_mask, rest_mask, envelope, threshold = detect_active_regions(emg)
    return SessionSummary(
        session=session_id,
        condition=meta["condition"],
        label=meta["label"],
        emg=emg,
        time=time,
        active_mask=active_mask,
        rest_mask=rest_mask,
        envelope=envelope,
        threshold=threshold,
    )

In [4]:
session_summaries: Dict[int, SessionSummary] = {
    session_id: build_session_summary(session_id)
    for session_id in SESSION_METADATA.index
}

metrics_df = pd.concat(
    [summary.channel_metrics() for summary in session_summaries.values()],
    ignore_index=True,
)
metrics_df.head()

AttributeError: 'Series' object has no attribute 'mad'

In [ ]:
session_level_summary = pd.DataFrame([
    {
        "session": summary.session,
        "condition": summary.condition,
        "label": summary.label,
        "samples": summary.emg.shape[0],
        "active_duration_s": summary.active_duration,
        "rest_duration_s": summary.rest_duration,
        "active_fraction": summary.active_duration / max((summary.emg.shape[0] / FS_HZ), 1e-6),
    }
    for summary in session_summaries.values()
]).sort_values(["condition", "session"])
session_level_summary

In [ ]:
condition_metrics = (
    metrics_df.groupby(["condition", "channel"], observed=True)
    .agg(
        rms_active_mean=("rms_active", "mean"),
        rms_active_median=("rms_active", "median"),
        rms_rest_mean=("rms_rest", "mean"),
        activation_ratio_median=("activation_ratio", "median"),
        peak_active_mean=("peak_active", "mean"),
    )
    .reset_index()
)
condition_metrics.head()

In [ ]:
def plot_stack_comparison(summary: SessionSummary, max_duration: float = 8.0, normalize: bool = True):
    samples = min(summary.emg.shape[0], int(max_duration * FS_HZ))
    emg = summary.emg[:samples]
    time = summary.time[:samples]
    offset = 1.0 if normalize else emg.std(axis=0).max()
    fig, ax = plt.subplots(figsize=(16, 10))
    for idx, channel in enumerate(CHANNEL_IDS):
        trace = emg[:, idx]
        if normalize:
            denom = np.ptp(trace)
            if denom == 0:
                denom = 1.0
            trace = (trace - trace.min()) / denom
        ax.plot(time, trace + idx * offset, linewidth=0.7, alpha=0.9)
    ax.set_title(f"Session {summary.session:02d} – {summary.label}")
    ax.set_xlabel("Time (s)")
    ax.set_yticks([i * offset for i in CHANNEL_IDS])
    ax.set_yticklabels([f"Ch {ch}" for ch in CHANNEL_IDS])
    ax.set_xlim(time[0], time[-1])
    ax.grid(True, axis="x", alpha=0.2)
    plt.tight_layout()
    return fig


def plot_envelope_with_activity(summary: SessionSummary, max_duration: float = 20.0):
    samples = min(summary.emg.shape[0], int(max_duration * FS_HZ))
    time = summary.time[:samples]
    envelope = summary.envelope[:samples]
    active_mask = summary.active_mask[:samples]
    threshold = summary.threshold
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(time, envelope, label="Envelope", color="steelblue")
    ax.fill_between(time, 0, envelope, where=active_mask, color="orangered", alpha=0.3, label="Active")
    ax.axhline(threshold, color="black", linestyle="--", linewidth=1.0, label=f"Threshold = {threshold:.2f}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Envelope (RMS)")
    ax.set_title(f"Session {summary.session:02d} – Activity envelope")
    ax.legend(loc="upper right")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    return fig

In [ ]:
def plot_condition_heatmap(matrix: np.ndarray, condition: str, metric_label: str, cmap: str = "viridis"):
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(matrix, cmap=cmap, aspect="auto")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(
                j, i, f"{CHANNEL_GRID[i, j]}",
                ha="center", va="center", color="white", fontsize=10,
                fontweight="bold",
            )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{condition} – {metric_label}")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(metric_label)
    plt.tight_layout()
    return fig


def condition_metric_grid(condition_metrics: pd.DataFrame, metric: str) -> Dict[str, np.ndarray]:
    results = {}
    for condition, subset in condition_metrics.groupby("condition", observed=True):
        pivot = subset.pivot(index="channel", columns="condition", values=metric)
        series = pivot[condition].reindex(CHANNEL_IDS)
        results[condition] = series.to_numpy().reshape(CHANNEL_GRID.shape)
    return results


def plot_condition_boxplot(metrics_df: pd.DataFrame, metric: str, title: str):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(
        data=metrics_df,
        x="condition",
        y=metric,
        order=CONDITION_ORDER,
        ax=ax,
    )
    sns.stripplot(
        data=metrics_df,
        x="condition",
        y=metric,
        order=CONDITION_ORDER,
        ax=ax,
        color="black",
        size=3,
        alpha=0.5,
    )
    ax.set_title(title)
    ax.set_xlabel("Condition")
    ax.set_ylabel(metric)
    plt.tight_layout()
    return fig

In [ ]:
stack_figures = []
for session_id in [0, 5, 8, 11]:
    stack_figures.append(plot_stack_comparison(session_summaries[session_id]))
stack_figures[:2]

In [ ]:
envelope_figures = []
for session_id in [0, 5, 8, 11]:
    envelope_figures.append(plot_envelope_with_activity(session_summaries[session_id]))
envelope_figures[:2]

In [ ]:
condition_heatmaps = condition_metric_grid(condition_metrics, "rms_active_mean")
heatmap_figures = []
for condition in CONDITION_ORDER:
    if condition in condition_heatmaps:
        matrix = condition_heatmaps[condition]
        heatmap_figures.append(plot_condition_heatmap(matrix, condition, "Mean active RMS"))
heatmap_figures[:2]

In [ ]:
boxplot_fig = plot_condition_boxplot(
    metrics_df.dropna(subset=["activation_ratio"]),
    metric="activation_ratio",
    title="Activation ratio distribution (active RMS / rest RMS)",
)
boxplot_fig

In [ ]:
try:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
except ImportError:
    PCA = None
    StandardScaler = None



def build_active_window_matrix(
    summaries: Iterable[SessionSummary],
    window_sec: float = 0.250,
    step_sec: float = 0.125,
    zscore: bool = True,
    max_windows_per_session: int = 400,
):
    """Return feature matrix & labels by sliding window across active segments."""
    window = max(int(window_sec * FS_HZ), 1)
    step = max(int(step_sec * FS_HZ), 1)
    features: List[np.ndarray] = []
    labels: List[str] = []
    for summary in summaries:
        emg = summary.emg
        mask = summary.active_mask
        idx = 0
        windows = 0
        while idx + window <= emg.shape[0] and windows < max_windows_per_session:
            segment = emg[idx : idx + window]
            if mask[idx : idx + window].mean() < 0.8:
                idx += step
                continue
            feat = np.sqrt(np.mean(np.square(segment), axis=0))
            # Concatenate basic pair features (optional expansions)
            features.append(feat)
            labels.append(summary.condition)
            windows += 1
            idx += step
    if not features:
        return np.empty((0, len(CHANNEL_IDS))), []
    X = np.vstack(features)
    if zscore and StandardScaler is not None:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    return X, labels



if PCA is not None:
    X_active, y_active = build_active_window_matrix(session_summaries.values())
    if X_active.shape[0] >= 3:
        pca = PCA(n_components=3, random_state=42)
        X_pca = pca.fit_transform(X_active)
        pca_df = pd.DataFrame({
            "PC1": X_pca[:, 0],
            "PC2": X_pca[:, 1],
            "PC3": X_pca[:, 2],
            "condition": y_active,
        })
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(111, projection="3d")
        for condition in CONDITION_ORDER:
            subset = pca_df[pca_df["condition"] == condition]
            ax.scatter(
                subset["PC1"], subset["PC2"], subset["PC3"],
                label=condition, alpha=0.6, s=20,
            )
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.set_zlabel("PC3")
        ax.set_title("PCA of active windows (RMS features)")
        ax.legend()
        plt.tight_layout()
        pca_fig = fig
    else:
        X_active = np.empty((0, len(CHANNEL_IDS)))
        y_active = []
        pca_fig = None
else:
    X_active = np.empty((0, len(CHANNEL_IDS)))
    y_active = []
    pca_fig = None
pca_fig

## Next steps

- Integrate glove IMU/pose data (if available) to correlate EMG activation with finger motion.
- Add overshoot rejection (e.g., Hampel filter or percentile clipping) before RMS computation.
- Explore frequency-domain features (median frequency shifts) to assess muscle fatigue across conditions.
- Expand comparative analysis to other subjects (S1–S4) using the same workflow.
